# The Optuna Master Template for CatBoost

*Companion to* **Mastering CatBoost — Pro Edition**, Chapter 4: Tuning & Diagnostics as System Control, §4.2 (The Tuning Philosophy: The Hierarchy of Needs) and §4.3 (The Stability Triangle).

This is the notebook you copy into real projects. It encodes Chapter 4's tuning discipline into a reusable Optuna + CatBoost harness: a search space organized by the Hierarchy of Needs, conditional parameters that respect CatBoost's compatibility rules (`bagging_temperature` vs. `subsample`), a dependency-free pruning callback, early stopping inside every trial, and a clean refit-and-report finish. Every design choice below is annotated with the book section it comes from.

In [ ]:
%pip install -q catboost==1.2.10 scikit-learn==1.6.1 numpy==2.0.2 pandas==2.2.2 matplotlib==3.10.0 optuna==4.2.1

import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# The progress bar is disabled below, so an absent Jupyter widget is immaterial.
warnings.filterwarnings("ignore", message="IProgress not found.*")
import optuna
from catboost import CatBoostClassifier, Pool, cv as catboost_cv

# The pinned experimental sampler options are disclosed in Section 4 below.
warnings.filterwarnings(
    "ignore",
    message=r"Argument ``(?:multivariate|group)`` is an experimental feature.*",
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

## 0. Level 0 first: the contract (§4.2.1)

Before any trial runs, the book's non-negotiables:

- **Leakage control** — temporal data gets chronological splits (`has_time=True` where relevant); here the data is i.i.d. synthetic, so a stratified split is legitimate.
- **Feature types** — categorical columns go to `cat_features`, never external one-hot.
- **Determinism** — one `RANDOM_SEED` everywhere. "Tuning is a scientific experiment; you cannot optimize a moving target."

We build a synthetic tabular dataset with genuine categorical structure so the Categorical Engine has work to do, and we carve out **three** sets: train (trials fit here), validation (early stopping + trial objective), and test (touched exactly once, at the very end).

In [ ]:
from sklearn.datasets import make_classification
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

n = 12_000
rng = np.random.RandomState(RANDOM_SEED)
X_num, y = make_classification(n_samples=n, n_features=15, n_informative=8,
                               n_redundant=3, flip_y=0.08, random_state=RANDOM_SEED)

# Two informative categoricals: a 30-level region effect and a 6-level device effect.
region = rng.randint(0, 30, n)
device = rng.randint(0, 6, n)
region_eff = rng.normal(0, 0.8, 30)
device_eff = rng.normal(0, 0.5, 6)
flip_up = (rng.rand(n) < 1 / (1 + np.exp(-(region_eff[region] + device_eff[device])))).astype(int)
y = np.where(rng.rand(n) < 0.35, flip_up, y)   # blend categorical signal into the target

X = pd.DataFrame(X_num, columns=[f"num_{i}" for i in range(15)])
X["region"] = pd.Series(region).astype(str)
X["device"] = pd.Series(device).astype(str)
CAT_FEATURES = ["region", "device"]

X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_trainval, y_trainval, test_size=0.25, random_state=RANDOM_SEED, stratify=y_trainval)

train_pool = Pool(X_train, y_train, cat_features=CAT_FEATURES)
valid_pool = Pool(X_valid, y_valid, cat_features=CAT_FEATURES)
print(f"train {X_train.shape} | valid {X_valid.shape} | test {X_test.shape} | pos rate {y.mean():.3f}")

## 1. Search-space design by the Hierarchy of Needs (§4.2.2–§4.2.4)

Principles encoded below:

- **Do not tune `iterations`** (§4.2.2). Overcommit (`ITERATIONS_CEILING`) and let early stopping find the stop — the Autopilot of §4.4.
- **Log-scale for multiplicative parameters**: `learning_rate` and `l2_leaf_reg` act multiplicatively, so sample them log-uniformly.
- **Level 2 (capacity)**: `depth` over the book's working range 4–10; remember each +1 doubles the leaves of a symmetric tree (§4.3.2).
- **Level 3 (robustness), conditionally**: `bootstrap_type` is a categorical choice, and its strength parameter differs per branch — `bagging_temperature` only under `Bayesian`, `subsample` only under `Bernoulli`/`MVS` (§4.2.4 Common Pitfall). Optuna's define-by-run API expresses this conditionality naturally.
- `random_strength`, `border_count`, `one_hot_max_size`, `rsm` complete the space; on narrow data `rsm` matters little (§4.2.4), but the template includes it because the template is meant for *your* wider datasets.

In [ ]:
ITERATIONS_CEILING = 800     # overcommit; early stopping decides the real count (Section 4.4)
EARLY_STOPPING = 50
EVAL_METRIC = "AUC"

def suggest_params(trial):
    params = {
        # --- Level 1: dynamics ---
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.3, log=True),
        # --- Level 2: capacity ---
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 0.5, 30.0, log=True),
        # --- Level 3: robustness ---
        "random_strength": trial.suggest_float("random_strength", 0.0, 5.0),
        "rsm": trial.suggest_float("rsm", 0.5, 1.0),
        # --- Data layer ---
        "border_count": trial.suggest_categorical("border_count", [32, 64, 128, 254]),
        "one_hot_max_size": trial.suggest_categorical("one_hot_max_size", [2, 10, 25]),
    }
    # Conditional branch: strength parameter depends on the bootstrap family (Section 4.2.4).
    bootstrap = trial.suggest_categorical("bootstrap_type", ["Bayesian", "Bernoulli", "MVS"])
    params["bootstrap_type"] = bootstrap
    if bootstrap == "Bayesian":
        params["bagging_temperature"] = trial.suggest_float("bagging_temperature", 0.0, 5.0)
    else:
        params["subsample"] = trial.suggest_float("subsample", 0.5, 1.0)
    return params

## 2. A dependency-free pruning callback

Optuna pruners can reduce compute by stopping trials whose intermediate validation ranks look unpromising. They can also discard late improvers. This template treats pruning as a measured compute-quality tradeoff, not a free saving.

In [ ]:
class OptunaPruningCallback:
    # Reports the eval metric to Optuna each `period` iterations and stops training when pruned.
    def __init__(self, trial, metric=EVAL_METRIC, eval_set_name="validation", period=25):
        self.trial = trial
        self.metric = metric
        self.eval_set_name = eval_set_name
        self.period = period
        self.pruned = False

    def after_iteration(self, info):
        it = info.iteration
        if it % self.period != 0:
            return True
        history = info.metrics.get(self.eval_set_name, {}).get(self.metric)
        if not history:
            return True
        self.trial.report(history[-1], step=it)
        if self.trial.should_prune():
            self.pruned = True
            return False   # stop training; we raise TrialPruned outside fit()
        return True

## 3. The objective (§4.2.2 + §4.4 in one function)

Everything Chapter 4 prescribes for a single experiment, in order: fixed seed, overcommitted iterations, eval set, `early_stopping_rounds` (which forces `od_type='Iter'` and discards any `od_pval` — §4.4.1), `use_best_model=True` so the trial's model is truncated to its best iteration, and the trial value taken from the best validation score. We also stash the stopped iteration count as a user attribute — you will want it when choosing the final training budget.

In [ ]:
def objective(trial):
    params = suggest_params(trial)
    model = CatBoostClassifier(
        iterations=ITERATIONS_CEILING,
        eval_metric=EVAL_METRIC,
        loss_function="Logloss",
        random_seed=RANDOM_SEED,
        verbose=0,
        allow_writing_files=False,
        **params,
    )
    pruning_cb = OptunaPruningCallback(trial)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        early_stopping_rounds=EARLY_STOPPING,   # forces od_type='Iter' (Section 4.4.1)
        use_best_model=True,
        callbacks=[pruning_cb],
    )
    if pruning_cb.pruned:
        raise optuna.TrialPruned()
    trial.set_user_attr("best_iteration", model.get_best_iteration())
    return model.get_best_score()["validation"][EVAL_METRIC]

## 4. Run the study

Best-practice details:

- **`TPESampler(seed=...)`** for reproducible Bayesian search; `multivariate=True, group=True` models interactions within compatible decomposed search-space groups, including the conditional bootstrap branches used here. It does not make the search exhaustive or guarantee a better candidate. Both options are experimental in pinned Optuna 4.2.1, so revalidate behavior before upgrading.
- **`MedianPruner`** with warmup so the first trials establish a baseline before anything gets killed.
- **`enqueue_trial`** ensures the book's baseline recipe (`depth=6, l2_leaf_reg=3, learning_rate=0.05` — the First Run Recipe of §4.2.4) is evaluated alongside the proposed candidates.
- `n_trials` is modest here to stay inside Colab's free-CPU budget; in real projects raise it and add `timeout`.

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(
        seed=RANDOM_SEED, multivariate=True, group=True
    ),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2),
    study_name="catboost_master_template",
)

# Seed with the book's First Run Recipe baseline (Section 4.2.4).
study.enqueue_trial({
    "learning_rate": 0.05, "depth": 6, "l2_leaf_reg": 3.0,
    "random_strength": 1.0, "rsm": 1.0, "border_count": 254,
    "one_hot_max_size": 2, "bootstrap_type": "Bernoulli", "subsample": 0.8,
})

t0 = time.perf_counter()
study.optimize(objective, n_trials=25, show_progress_bar=False)
print(f"25 trials in {time.perf_counter() - t0:.0f}s")
print(f"best validation {EVAL_METRIC}: {study.best_value:.4f}")
print("best params:")
for k, v in study.best_params.items():
    print(f"  {k} = {v}")
print("best trial stopped at iteration:", study.best_trial.user_attrs.get("best_iteration"))

In [ ]:
# Which parameters actually mattered on THIS dataset? (Chapter 4, Exercise 1)
importances = optuna.importance.get_param_importances(study)
imp = pd.Series(importances).sort_values()
imp.plot.barh(figsize=(7, 4), color="steelblue")
plt.title("Optuna parameter importances (this study, this dataset)")
plt.xlabel("importance"); plt.tight_layout(); plt.show()

hist = study.trials_dataframe(attrs=("number", "value", "state"))
best_so_far = hist["value"].cummax()
plt.figure(figsize=(7, 3.5))
plt.plot(hist["number"], hist["value"], "o", alpha=0.5, label="trial value")
plt.plot(hist["number"], best_so_far, "-", color="crimson", label="best so far")
plt.xlabel("trial"); plt.ylabel(f"valid {EVAL_METRIC}"); plt.legend(); plt.tight_layout(); plt.show()

## 5. Variant: cross-validated development objective

In [ ]:
def objective_cv(trial):
    params = suggest_params(trial)
    cv_results = catboost_cv(
        Pool(X_trainval, y_trainval, cat_features=CAT_FEATURES),
        params={**params, "iterations": 300, "eval_metric": EVAL_METRIC,
                "loss_function": "Logloss", "random_seed": RANDOM_SEED},
        fold_count=3, early_stopping_rounds=EARLY_STOPPING,
        verbose=False, as_pandas=True)
    return cv_results[f"test-{EVAL_METRIC}-mean"].max()

# This development variant executes before the final test opening below.
study_cv = optuna.create_study(
    direction="maximize", sampler=optuna.samplers.TPESampler(seed=RANDOM_SEED))
study_cv.optimize(objective_cv, n_trials=5)
print(f"CV development demo - best 3-fold {EVAL_METRIC}: {study_cv.best_value:.4f}")

## 6. Refit with frozen counts and open test once

In [ ]:
best_params = dict(study.best_params)
tuned_trees = int(study.best_trial.user_attrs.get("best_iteration") or
                  (ITERATIONS_CEILING - 1)) + 1

# Give the default candidate its own development-selected tree count.
baseline_dev = CatBoostClassifier(
    iterations=ITERATIONS_CEILING, eval_metric=EVAL_METRIC,
    loss_function="Logloss", random_seed=RANDOM_SEED,
    verbose=0, allow_writing_files=False)
baseline_dev.fit(train_pool, eval_set=valid_pool,
                 early_stopping_rounds=EARLY_STOPPING, use_best_model=True)
baseline_trees = baseline_dev.get_best_iteration() + 1

trainval_pool = Pool(X_trainval, y_trainval, cat_features=CAT_FEATURES)
final_model = CatBoostClassifier(
    iterations=tuned_trees, eval_metric=EVAL_METRIC, loss_function="Logloss",
    random_seed=RANDOM_SEED, verbose=0, allow_writing_files=False,
    **best_params)
baseline = CatBoostClassifier(
    iterations=baseline_trees, eval_metric=EVAL_METRIC, loss_function="Logloss",
    random_seed=RANDOM_SEED, verbose=0, allow_writing_files=False)
final_model.fit(trainval_pool)
baseline.fit(trainval_pool)

# Candidates and tree counts are frozen; now open test once for both.
test_auc = roc_auc_score(y_test, final_model.predict_proba(X_test)[:, 1])
base_auc = roc_auc_score(y_test, baseline.predict_proba(X_test)[:, 1])
print({"tuned_trees": tuned_trees, "baseline_trees": baseline_trees,
       "tuned_test_auc": round(test_auc, 4),
       "baseline_test_auc": round(base_auc, 4),
       "paired_test_difference": round(test_auc - base_auc, 4)})

## 7. Production checklist

- Freeze the study, candidate rule, and each development-selected iteration count before opening test.
- Persist seeds, resolved parameters, wall time, and trial outcomes.
- Bound parallelism and use chronological objectives for temporal targets.
- Treat pruning as a tradeoff: intermediate ranks can mislead.

## Takeaways

- Search-space hierarchy is workflow guidance. Iterations may be selected by early stopping, fixed-budget comparison, or explicit tuning according to the production constraint.
- Encode CatBoost compatibility rules conditionally.
- Enqueue the production candidate so no-change remains possible.
- Parameter importance and tuning lift are dataset- and protocol-specific.

## Exercises

**Exercise 1 - Pruning economics.** Run matched pruned and unpruned studies across sampler seeds. Report wall time, best development value, and late improvers; do not assume pruning is free.

**Exercise 2 - Fixed backbone.** Compare a fixed backbone with the conditional search under matched evaluation and wall-clock budgets. Report the selected development result without prescribing a winner.

**Exercise 3 - Seed sensitivity before test.** Refit tuned and default candidates across seeds on development data, each with its own validation-selected tree count. Freeze the decision rule before the one-shot test cell.

### Hidden solutions

<details><summary>Solution</summary>

**Exercise 1.** Create matched studies with the same sampler seeds and objective, changing only `NopPruner` versus the declared pruner. Store wall time, best validation value, prune fraction, and every pruned trial's last intermediate rank. Rerun pruned configurations without pruning to count late improvers; this directly measures the time/quality trade-off.

</details>

<details><summary>Solution</summary>

**Exercise 2.** Freeze the same train/validation folds and total budget. In the fixed-backbone study tune only the declared small parameter set; in the conditional study expose the larger valid branches. Compare best validation value, selected tree count, wall time, and between-seed variability. Complexity is justified only by a reproducible gain or operational benefit.

</details>

<details><summary>Solution</summary>

**Exercise 3.** Before opening test, refit the tuned and default recipes across model seeds, each with its own validation-selected iteration. Predeclare a promotion rule such as a minimum paired validation gain that exceeds seed variability. Freeze the winning recipe and its refit rule, then evaluate one final model once on test.

</details>
